In [17]:
import os

import pandas as pd
from dotenv import load_dotenv
import folium
import polyline
import requests
import urllib3

# Config

In [18]:
HTTP_PROXY = 'http://sia-lb.telekom.de:8080'
HTTPS_PROXY = 'http://sia-lb.telekom.de:8080'

os.environ['http_proxy'] = HTTP_PROXY
os.environ['https_proxy'] = HTTPS_PROXY

activities_url = 'https://www.strava.com/api/v3/athlete/activities'
auth_url = 'https://www.strava.com/oauth/token'

In [19]:
load_dotenv()

CLIENT_ID = os.getenv('CLIENT_ID')
CLIENT_SECRET = os.getenv('CLIENT_SECRET')
REFRESH_TOKEN = os.getenv('REFRESH_TOKEN')
GRANT_TYPE = os.getenv('GRANT_TYPE')

# Get data from Strava API

In [20]:
payload = {
    'client_id': CLIENT_ID,
    'client_secret': CLIENT_SECRET,
    'refresh_token': REFRESH_TOKEN,
    'grant_type': GRANT_TYPE,
    'f': 'json',
}

print('Requesting Token...\n')
res = requests.post(auth_url, data=payload, verify=False)
access_token = res.json()['access_token']
print(f'Access Token = {access_token} \n')

header = {'Authorization': 'Bearer ' + access_token}


request_page_num = 1
all_activities = []

while True:
    param = {'per_page': 200, 'page': request_page_num}
    my_dataset = requests.get(activities_url, headers=header, params=param).json()

    if len(my_dataset) == 0:
        break

    if all_activities:
        all_activities.extend(my_dataset)

    else:
        all_activities = my_dataset

    request_page_num += 1

print('The length of all activies is: ', len(all_activities))

Requesting Token...

Access Token = d19971ce7da87df8f04e4e15c9e5c62d1528f62a 

The length of all activies is:  193


In [22]:
df_all_activities = pd.json_normalize(all_activities)
df_all_activities.head()

,resource_state,name,distance,moving_time,elapsed_time,total_elevation_gain,type,sport_type,workout_type,id,...,external_id,from_accepted_tag,pr_count,total_photo_count,has_kudoed,athlete.id,athlete.resource_state,map.id,map.summary_polyline,map.resource_state
0,2,Hill Intervals,7502.8,2402,2405,230.0,Run,Run,0.0,15309238742,...,garmin_ping_465048251379,False,0,0,False,133094316,1,a15309238742,kcluHkq}j@DMn@yAJ]|@{B?i@Em@QwAIkB?GHW@Om@kIe@...,2
1,2,Easy Run,10090.9,3532,3579,83.0,Run,Run,0.0,15298830273,...,garmin_ping_464720550858,False,0,0,False,133094316,1,a15298830273,qeluHwm|j@NzBB`CIhAIdBKp@QnCWfB]~@s@bCk@bCOz@K...,2
2,2,First 100k,100376.0,14606,15953,1206.0,Ride,Ride,10.0,15287917307,...,garmin_ping_464391433871,False,11,0,False,133094316,1,a15287917307,waluH_q|j@rN`EnCbJoi@n}AqPjjA{L`g@cGfBkJqKeE}@...,2
3,2,Tempo Run,12146.8,3056,3059,125.0,Run,Run,0.0,15272351610,...,garmin_ping_463957509479,False,0,0,False,133094316,1,a15272351610,qeluHim|j@JhFAv@ObCIj@[lEKn@]~@Ux@[z@Qr@YnAYpB...,2
4,2,Basic Run,8563.2,2643,2643,78.0,Run,Run,0.0,15252182113,...,garmin_ping_463303625408,False,0,0,False,133094316,1,a15252182113,ywluHoq|j@FpAHp@HpBFv@ELMDI?e@QOOa@i@O]c@q@sAs...,2


In [27]:
df_all_activities[['id', 'name', 'total_photo_count', 'photo_count', 'start_date_local']][
    df_all_activities['total_photo_count'] > 0
].head(10)

,id,name,total_photo_count,photo_count,start_date_local
150,13591252207,Après Ski Run,1,0,2025-02-10T16:29:04Z
166,13316805259,Afternoon Run,1,0,2025-01-10T15:10:06Z
169,13278309978,Morning Run,1,0,2025-01-06T07:17:54Z


In [ ]:
unimportant_columns = ['resource_state', 'photo_count']


In [ ]:
df_important = df_all_activities[important_cols]
df_important.head()

# Plot polyline on Folium Map

In [ ]:
# Deine Polyline-Zeichenkette
polyline_string = 'waluH_q|j@rN`EnCbJoi@n}AqPjjA{L`g@cGfBkJqKeE}@uTvOiB`J~B`Uk@|KuKvSeLxb@uhA~kB{CrIm@dI`AtM|Yl_Bsh@`Oc`@Pq[|Jeb@b@eLsF{Zgd@aN{FmLhc@iBjWi}@j\\{j@woAyX_f@cq@cvAiNwXsAuIt@cf@tD{Lx@wKsBg\\cFwXgCuDaI{B}IwJaSqd@{VmQxW_VjEgZzEabBc@_JuEoP`CqJjBiVbKsDfNoMrBmRpE_NtHyc@jPw[lXwm@zDuOvWkB~CaFfDeh@}Lea@nCgNNeScF_PmAmU`EoQaFw{@v@m^lAwLfCkH`PyMvJkQfCkSgFsB}H~CsIuQeJ|A_B~FuA|TeWr_@uFrPyNfNiDhAuOqMkWkHyKfFkGbMoKjA_DiC_IkQcU}iAuLiX[uLeI_NcDaOuPi]oJeP{H_DkEsFyCsG_DeMe@gDeJ_FtIxErCrFbChNzJxMcK{NoBkMcB}@q@uDaO{KuF~L_C]_EwL`@oGwDyBkH}_@_GcIaDlGcDwGyOwo@oNkX{BeKiPs]_CmK_NsOwHeYcNoLcIl@{G{BiLeM}IoUs]q_Bk@uLAkD|AbAf@fEhEzGdBM~@uDx@wT_AgIpCuMl@oWdFwLm@qN~AyBx@gPkI}V_Fw[wEaRzKwG`MlBlLiAdDeJpFhAbBsH|K|@fHoFx^~ElKr_@nMtJnD_AnO}NhR_BuFrc@|@l]j`AlnCdTxX`E~SzEgG~KRnJvKzHX|CxHlKxC`D[jD{FbMuBjEpEbSoDxIjJ~ImMxViMnKbC~DeDhEzHjG}BpDdHxE~AxF`ItJx`@xMas@fIuNpOmp@~@_QvM}YdIvAjXr[pOaChM_Q`N}CrJkOxZ{Tr\\_q@nBsRdJ`LaDmGnEmb@}B}^x@{HdIvMrt@tJvWrTrK`D~ExEzFdOfF`f@u@zn@lGbR|C~b@sCpVFpQtFn[r@xQjHbP]bVtB~NzFbDvQ}MdH`B|Sd\\|DhZhSlRdKvEbDrP|IxF|ElIrApEhAhVh]b`@nAbFzApU~r@wA~ElDg@pu@gMzc@_E|l@NxKjDbQA|JuJ~j@sCfl@~M|o@jDtHlKxKpCxKI|K_G~Xx@|VyEnVhFlVvHd|@cFnb@wGfOuF|EuXnKUtBeT~QyLtReQdA}IdEyDrGsKx^{bAboAuZrTsX`KcKhLsThGgb@nX{BxFbUfKdMcCjS|StKjPlF}IzNdGdXk@pNlKtEInVeOzFsLlC{OxI{O`Lp@tXeJnCiFyAcThDsPx@wS'

# 1. Polyline-String in eine Liste von Koordinaten (Latitude, Longitude) dekodieren
decoded_coords = polyline.decode(polyline_string)

# 2. Den Startpunkt der Route finden, um die Karte zu zentrieren
if decoded_coords:
    center_point = decoded_coords[0]
else:
    print('Die Polyline ist leer. Es können keine Koordinaten dekodiert werden.')

# 3. Eine Folium-Karte erstellen
# Der Parameter 'zoom_start' bestimmt den anfänglichen Zoom-Level der Karte
m = folium.Map(location=center_point, zoom_start=13)

# 4. Die Polyline zur Karte hinzufügen
folium.PolyLine(locations=decoded_coords, color='blue', weight=5, opacity=0.8).add_to(m)

# 5. Optional: Start- und Endpunkt der Route markieren
folium.Marker(
    location=decoded_coords[0], popup='Startpunkt', icon=folium.Icon(color='green')
).add_to(m)
folium.Marker(
    location=decoded_coords[-1], popup='Endpunkt', icon=folium.Icon(color='red')
).add_to(m)

# 6. Die Karte in der Notebook-Zelle anzeigen
m